In [1]:
import osmnx as ox
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import folium
from shapely.geometry import LineString, Point
from shapely.ops import unary_union
import networkx as nx
import os
from branca.colormap import LinearColormap

In [2]:
operation_crs = "EPSG:2056"  # Swiss coordinate system
target_crs = "EPSG:4326"  # WGS84 coordinate system

print("Set parameters : GG or GE, bike or walk")
territory = 'GG' # GG or GE
network = "walk"  # walk or bike
index_col = f"{network}_index" 

input_file_path = '../../Data/input'
output_step1_path=f"../../Data/output/{network}/{territory}/step-1"
output_step2_path=f"../../Data/output/{network}/{territory}/step-2"
output_step3_path=f"../../Data/output/{network}/{territory}/step-3_rural"
save_path = f"../../Data/output/{network}/{territory}/step-3_rural"

#import file
print("Import files ...")
all_network = gpd.read_parquet(f'{output_step1_path}/step1_all_segments.parquet')

index_scores = gpd.read_parquet(f'{output_step3_path}/step3_index.parquet')


Set parameters : GG or GE, bike or walk
Import files ...


In [3]:
#Check colomns
index_scores.head()


,segment_id,geometry,length,lac_cours_deau,fontaine,espaces_ouverts,rez_actif,tp,amenite,temperature,...,vitesse,eclairage,walk_index,walk_index_unweighted,rd_hors_agglo_masked,rd_hors_agglo_overlap_ratio,Classe_Attractivité,Classe_Commodité,Classe_Infrastructure,Classe_Sécurité
0,000000,"LINESTRING (6.15792 46.20527, 6.15782 46.20537...",33.141,1.0,0.0,0.0,0.75,1.0,0.5,0.0492,...,0.3509,1.0,0.3880,0.5173,False,NaN,0.619,0.2427,0.3506,0.2851
1,000001,"LINESTRING (6.15767 46.20551, 6.15777 46.20542...",33.141,1.0,0.0,0.0,0.75,1.0,0.5,0.0492,...,0.3509,1.0,0.3880,0.5173,False,NaN,0.619,0.2427,0.3506,0.2851
2,000002,"LINESTRING (5.95491 46.21124, 5.95495 46.21126...",50.000,0.0,0.0,0.0,0.00,0.0,0.0,0.0000,...,0.1482,0.0,0.0393,0.2000,False,NaN,0.000,0.1987,0.3226,0.2001
3,000003,"LINESTRING (5.95538 46.21155, 5.95577 46.21181...",50.000,0.0,0.0,0.0,0.00,0.0,0.0,0.0000,...,0.0000,0.0,0.0393,0.2248,False,NaN,0.000,0.3249,0.3104,0.1688
4,000004,"LINESTRING (5.95584 46.21186, 5.95624 46.21222)",50.000,0.0,0.0,0.0,0.00,0.0,0.0,0.0000,...,0.0000,0.0,0.0393,0.2261,False,NaN,0.000,0.3249,0.3157,0.1688


In [4]:
# Check if 'segment_id' exists in both
assert 'segment_id' in all_network.columns
assert 'segment_id' in index_scores.columns

# Convert both to string with 6-digit zero-padding
all_network['segment_id'] = all_network['segment_id'].astype(str).str.zfill(6)
index_scores['segment_id'] = index_scores['segment_id'].astype(int).astype(str).str.zfill(6)



In [5]:
# Merge
print("Add column (containing features count) to merge -->")
all_network = all_network.merge(index_scores[['segment_id', 'walk_index', 'Classe_comfort', 'Classe_infrastructure', 'Classe_securite']], on='segment_id', how='left')


Add column (containing features count) to merge -->


KeyError: "['Classe_comfort', 'Classe_infrastructure', 'Classe_securite'] not in index"

In [6]:
# Load pedestrian network attributes
# From SITG (shp)
print("Loading accidents involving pedestrians")
file_path = '/Users/Helo/Documents/Action_situee/index-marchabilite-ge/Data/input/accidents/OCT_ACCIDENTS-SHP'
accidents = gpd.read_file(f"{file_path}/OCT_ACCIDENTS.shp")

# Filter accidents that involve pedestrians
accidents["has_pedestrian"] = accidents["PIETONS"] > 0

accidents_pieton = accidents[accidents["has_pedestrian"] == True]

# Define the area (can be the city or the full canton boundary if you have it)
area_name = "Geneva, Switzerland"

# Load amenities GeoDataFrame from OSM
print("Loading amenities...")

tags = {
    "amenity": [
        "school", "university", "hospital", "clinic", "pharmacy",
        "restaurant", "cafe", "bar", "library", "bank", "post_office",
        "kindergarten", "theatre", "cinema", "place_of_worship"
    ]
}

# Download amenities
print(f"Fetching amenities for {area_name}")
amenities = ox.features_from_place(area_name, tags=tags)

Loading accidents involving pedestrians


DataSourceError: /Users/Helo/Documents/Action_situee/index-marchabilite-ge/Data/input/accidents/OCT_ACCIDENTS-SHP/OCT_ACCIDENTS.shp: No such file or directory

In [7]:
all_network_wgs = all_network.to_crs(epsg=4326)
#accidents_pieton_wsg = accidents_pieton.to_crs(epsg=4326)
#amenities_wsg = amenities.to_crs(epsg=4326)

# Define the colormap
import branca.colormap as cm

colormap = cm.linear.RdYlGn_09.scale(0, 1)
colormap.caption = 'Score Normalisé (0 = Mauvais, 1 = Très Bon)'

colormap_acc = LinearColormap(
    colors=["#E60000", "#ffffff"],  # dark red → white
    vmin=0, vmax=1
)
colormap_acc.caption = 'Score Normalisé (0 = Mauvais, 1 = Très Bon)'


def make_style_function(column_name, colormap):
    def style_function(feature):
        value = feature['properties'].get(column_name, None)
        return {
            'color': colormap(value) if value is not None else 'gray',
            'weight': 5,
            'opacity': 1
        }
    return style_function

import branca.colormap as cm


# Generate custom style functions
style_accident = make_style_function("N-RoadSafety", colormap_acc)
style_amenities = make_style_function("N-ProxAmenities", colormap)
style_index = make_style_function("walk_index", colormap)


In [8]:
# Compute center
center = all_network_wgs.geometry.unary_union.centroid
m = folium.Map(location=[center.y, center.x], zoom_start=13, tiles="cartodbpositron")

# --- Add pedestrian segments ---
print("Map network")
'''folium.GeoJson(
    all_network_wgs,
    name='Pedestrian Network',
    style_function=lambda x: {'color': '#3489db', 'weight': 2},
).add_to(m)'''

# ----- Mapping scores ----
'''# Add GeoJson layer
print("Mapping roadsafety ... ")
folium.GeoJson(
    all_network_wgs,
    name="Score accidents avec piétons",
    style_function=style_accident,
    tooltip=folium.GeoJsonTooltip(fields=["segment_id", "N-RoadSafety"], aliases=["ID", "Score Accidents"])
).add_to(m)

# Add GeoJson layer
print("Mapping prox amenities ... ")
folium.GeoJson(
    all_network_wgs,
    name="Score proximité aménités",
    style_function=style_amenities,
    tooltip=folium.GeoJsonTooltip(fields=["segment_id", "N-ProxAmenities"], aliases=["ID", "Score Proximité Amenités"])
).add_to(m)'''

# Add GeoJson layer
print("Mapping global index ... ")
folium.GeoJson(
    all_network_wgs,
    name="Indice de marchabilité",
    style_function=style_index,
    tooltip=folium.GeoJsonTooltip(fields=["segment_id", "walk_index"], aliases=["ID", "Indice"])
).add_to(m)

# ----- Mapping features -----
'''# Create a FeatureGroup for accidents
accidents_layer = folium.FeatureGroup(name="Accidents piétons", show=True)

print("Mapping accidents...")
for _, row in accidents_pieton_wsg.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=1,
        color='#1D3557',
        fill=True,
        fill_opacity=0.7,
        popup='Accident',
    ).add_to(accidents_layer)'''



# Create a FeatureGroup for amenities
'''amenities_layer = folium.FeatureGroup(name="Amenités", show=True)

# Loop through amenities and add them as small green markers
print("Mapping amenities...")
for _, row in amenities_wsg.iterrows():
    amenity_type = row.get("amenity", "Amenity")

    # Use point geometry or fallback to centroid
    geom = row.geometry
    if geom.geom_type == "Point":
        lat, lon = geom.y, geom.x
    else:
        centroid = geom.centroid
        lat, lon = centroid.y, centroid.x

    folium.CircleMarker(
        location=[lat, lon],
        radius=1,
        color='#567572',
        fill=True,
        fill_opacity=0.7,
        popup=folium.Popup(f"Type d'amenité: {amenity_type}", max_width=200),
    ).add_to(amenities_layer)

# Add the FeatureGroup to the map
accidents_layer.add_to(m)
amenities_layer.add_to(m)'''


# ---- Add legend -----
colormap.add_to(m)
colormap_acc.add_to(m)


# Show or save
folium.LayerControl().add_to(m)
#m.save("score_map.html")
m


/var/folders/rf/gj35881j7ydf0dq8kldyt7nw0000gn/T/ipykernel_16784/1953634943.py:2: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  center = all_network_wgs.geometry.unary_union.centroid


Map network
Mapping global index ... 


AssertionError: The field walk_index is not available in the data. Choose from: ('u', 'v', 'key', 'infra_bike', 'bike_direction', 'source', 'highway', 'cycleway', 'cycleway:left', 'cycleway:right', 'bicycle', 'oneway', 'oneway:bicycle', 'name', 'maxspeed', 'len_m', 'edge_id', 'length', 'surface', 'segment_id').

In [9]:
m.save("score_map.html")


AssertionError: The field walk_index is not available in the data. Choose from: ('u', 'v', 'key', 'infra_bike', 'bike_direction', 'source', 'highway', 'cycleway', 'cycleway:left', 'cycleway:right', 'bicycle', 'oneway', 'oneway:bicycle', 'name', 'maxspeed', 'len_m', 'edge_id', 'length', 'surface', 'segment_id').